# Pneumonia State Transition Demo

This notebook shows the hidden pneumonia state transition model in the same numeric style as the other Patient demos.

In [91]:
import os
import sys
from pathlib import Path

try:
    current_dir = Path(__file__).resolve().parent
except NameError:
    current_dir = Path.cwd().resolve()

project_root = current_dir
while project_root != project_root.parent and not (project_root / "Patient").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from Patient.models.pneumonia_model import PneumoniaStateTransitionModel
from Patient.models.base_model import EventType

In [92]:
import os
import sys
from pathlib import Path

try:
    current_dir = Path(__file__).resolve().parent
except NameError:
    current_dir = Path.cwd().resolve()

project_root = current_dir
while project_root != project_root.parent and not (project_root / "Patient").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from Patient.models.pneumonia_model import PneumoniaStateTransitionModel
from Patient.models.base_model import EventType

In [93]:
model = PneumoniaStateTransitionModel()
NO = model.NO_INTERVENTION
A = model.ANTIBIOTIC_A
B = model.ANTIBIOTIC_B

print(model.get_model_info())
print("NO =", NO)
print("A  =", A)
print("B  =", B)

{'model_name': 'PneumoniaStateTransitionModel', 'model_params': {'k': 3, 'u': 5, 'v': 6, 'p': 4, 'q': 5, 'm': 7, 'n': 8, 'type': 'pneumonia_state_transition'}, 'initial_state': (0, 0, 1), 'available_interventions': [(0, 0, 0), (0, 0, 1), (0, 1, 0)]}
NO = (0, 0, 0)
A  = (0, 0, 1)
B  = (0, 1, 0)


## Representative Intervention Paths

In [89]:
def run_scenario(name, initial_state, actions):
    result = model.simulate(actions, initial_state=initial_state, return_details=True)
    states = result["states"]
    events = result["events"]
    rows = []
    action_at_time = [NO] + actions
    event_at_time = ["initial"] + [event.value for event in events]
    for t, (action, state, event) in enumerate(zip(action_at_time, states, event_at_time)):
        rows.append({
            "scenario": name,
            "t": t,
            "action": action,
            "action_name": model.action_name(action),
            "state": state,
            "state_name": model.state_name(state),
            "event": event,
        })
    return rows

scenarios = {
    "mild_NO_to_severe": (model.MILD, [NO] * model.model_params["k"]),
    "mild_A_to_healthy": (model.MILD, [A] * model.model_params["u"]),
    "mild_B_to_healthy": (model.MILD, [B] * model.model_params["v"]),
    "severe_A_to_mild": (model.SEVERE, [A] * model.model_params["p"]),
    "severe_B_to_mild": (model.SEVERE, [B] * model.model_params["q"]),
    "severe_A_tolerance_then_B": (
        model.SEVERE,
        [A] * model.model_params["m"] + [B] * model.model_params["q"],
    ),
    "severe_B_tolerance_then_A": (
        model.SEVERE,
        [B] * model.model_params["n"] + [A] * model.model_params["p"],
    ),
}

all_rows = []
for name, (initial_state, actions) in scenarios.items():
    all_rows.extend(run_scenario(name, initial_state, actions))

try:
    import pandas as pd
    df = pd.DataFrame(all_rows)
    display(df)
except ImportError:
    for row in all_rows:
        print(row)

,scenario,t,action,action_name,state,state_name,event
0,mild_NO_to_severe,0,"(0, 0, 0)",none,"(0, 1, 0)",mild,initial
1,mild_NO_to_severe,1,"(0, 0, 0)",none,"(0, 1, 0)",mild,state_maintenance
2,mild_NO_to_severe,2,"(0, 0, 0)",none,"(0, 1, 0)",mild,state_maintenance
3,mild_NO_to_severe,3,"(0, 0, 0)",none,"(0, 0, 1)",severe,deterioration
4,mild_A_to_healthy,0,"(0, 0, 0)",none,"(0, 1, 0)",mild,initial
5,mild_A_to_healthy,1,"(0, 0, 1)",antibiotic_A,"(0, 1, 0)",mild,state_maintenance
6,mild_A_to_healthy,2,"(0, 0, 1)",antibiotic_A,"(0, 1, 0)",mild,state_maintenance
7,mild_A_to_healthy,3,"(0, 0, 1)",antibiotic_A,"(0, 1, 0)",mild,state_maintenance
8,mild_A_to_healthy,4,"(0, 0, 1)",antibiotic_A,"(0, 1, 0)",mild,state_maintenance
9,mild_A_to_healthy,5,"(0, 0, 1)",antibiotic_A,"(1, 0, 0)",healthy,complete_transition


## One Intervention-Switch Path

In [90]:
actions = [NO] * 2 + [A] * model.model_params["p"] + [B] * model.model_params["v"]
rows = run_scenario("severe_A_then_B", model.SEVERE, actions)

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except ImportError:
    for row in rows:
        print(row)

,scenario,t,action,action_name,state,state_name,event
0,severe_A_then_B,0,"(0, 0, 0)",none,"(0, 0, 1)",severe,initial
1,severe_A_then_B,1,"(0, 0, 0)",none,"(0, 0, 1)",severe,state_maintenance
2,severe_A_then_B,2,"(0, 0, 0)",none,"(0, 0, 1)",severe,state_maintenance
3,severe_A_then_B,3,"(0, 0, 1)",antibiotic_A,"(0, 0, 1)",severe,state_maintenance
4,severe_A_then_B,4,"(0, 0, 1)",antibiotic_A,"(0, 0, 1)",severe,state_maintenance
5,severe_A_then_B,5,"(0, 0, 1)",antibiotic_A,"(0, 0, 1)",severe,state_maintenance
6,severe_A_then_B,6,"(0, 0, 1)",antibiotic_A,"(0, 1, 0)",mild,partial_transition
7,severe_A_then_B,7,"(0, 1, 0)",antibiotic_B,"(0, 1, 0)",mild,state_maintenance
8,severe_A_then_B,8,"(0, 1, 0)",antibiotic_B,"(0, 1, 0)",mild,state_maintenance
9,severe_A_then_B,9,"(0, 1, 0)",antibiotic_B,"(0, 1, 0)",mild,state_maintenance
